In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix
import shap
import warnings
warnings.filterwarnings('ignore')

# -----------------------------
# 1. Load data
# -----------------------------
data_path = r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\insurance_data_cleaned.csv"
df = pd.read_csv(data_path, low_memory=False)

# Drop empty columns
df.drop(columns=['NumberOfVehiclesInFleet'], inplace=True, errors='ignore')

# -----------------------------
# 2. Create targets
# -----------------------------
target_reg = 'TotalClaims'  # Claim severity
# Binary target for classification
df['ClaimOccurred'] = (df[target_reg] > 0).astype(int)
target_cls = 'ClaimOccurred'

# -----------------------------
# 3. Features
# -----------------------------
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove targets from features
for t in [target_reg, target_cls]:
    if t in numerical_features:
        numerical_features.remove(t)
    if t in categorical_features:
        categorical_features.remove(t)

# Fix mixed-type categorical columns
for col in categorical_features:
    df[col] = df[col].astype(str)
    df[col] = df[col].replace('nan', np.nan)

# -----------------------------
# 4. Train/Test split
# -----------------------------
# Regression – only for policies with claims
df_reg = df[df[target_reg] > 0].copy()
X_reg = df_reg[numerical_features + categorical_features]
y_reg = df_reg[target_reg]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# Classification – all policies
X_cls = df[numerical_features + categorical_features]
y_cls = df[target_cls]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42
)

# -----------------------------
# 5. Preprocessing pipelines
# -----------------------------
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


In [11]:
# Regression results
for name, model in models_reg.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train_r, y_train_r)
    preds = pipeline.predict(X_test_r)
    rmse = np.sqrt(mean_squared_error(y_test_r, preds))
    r2 = r2_score(y_test_r, preds)
    print(f"{name}: RMSE={rmse:.2f}, R2={r2:.3f}")


LinearRegression: RMSE=27482.69, R2=0.520
RandomForest: RMSE=14390.65, R2=0.868
XGBoost: RMSE=14156.18, R2=0.873


In [ ]:
# -----------------------------
# Classification Models – Claim Occurrence
# -----------------------------
models_cls = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# -----------------------------
# Classification results
# -----------------------------
for name, model in models_cls.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train_c, y_train_c)
    preds = pipeline.predict(X_test_c)
    print(f"\n{name} Classification Report:")
    print(classification_report(y_test_c, preds))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_c, preds))



LogisticRegression Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    299147
           1       1.00      0.90      0.95       883

    accuracy                           1.00    300030
   macro avg       1.00      0.95      0.97    300030
weighted avg       1.00      1.00      1.00    300030

Confusion Matrix:
[[299147      0]
 [    84    799]]
